<a href="https://colab.research.google.com/github/hongxu-yn/Southeast-Asia-XCO2-STK/blob/main/src/step04_spatiotemporal_patterns.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Initialization

In [ ]:
from google.colab import drive
!pip install -q netCDF4 joblib tqdm xarray pykrige gstools cartopy scipy regionmask

drive.mount('/content/drive')
print("✅ Drive mounted successfully and environment initialization complete!")

In [ ]:
import yaml
import geopandas as gpd
from pathlib import Path
import os
import zipfile
import urllib.request
import warnings
from pathlib import Path

import numpy as np
import rasterio
from rasterio.enums import Resampling
import matplotlib as mpl        # <--- Add this line
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Circle, Patch, ConnectionPatch
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib.gridspec as gridspec
from shapely.geometry import box

warnings.filterwarnings("ignore")
mpl.rcParams.update({
    "font.size": 8,            # Global base font size
    "axes.labelsize": 8,       # Axis labels (xlabel, ylabel)
    "xtick.labelsize": 8,      # x-axis tick labels
    "ytick.labelsize": 8,      # y-axis tick labels
    "legend.fontsize": 8,      # Legend font size
    "axes.titlesize": 9,       # Subplot title (slightly larger for distinction)
    "axes.linewidth": 0.8,
    "xtick.direction": "out",
    "ytick.direction": "out"
})

plt.rcParams['axes.unicode_minus'] = False

def add_scalebar(ax, length_km=1000, location=(0.05, 0.05)):
    x0, x1, y0, y1 = ax.get_extent(ccrs.PlateCarree())
    lon_s, lat_s = x0 + (x1-x0)*location[0], y0 + (y1-y0)*location[1]
    lon_e = lon_s + (length_km / (111.32 * np.cos(np.deg2rad(lat_s))))
    kw = dict(transform=ccrs.PlateCarree(), color='k', zorder=20)
    ax.plot([lon_s, lon_e], [lat_s, lat_s], lw=2, **kw)
    tick_h = (y1-y0)*0.025
    for x in [lon_s, lon_e]:
        ax.plot([x, x], [lat_s, lat_s + tick_h], lw=2, **kw)
    ax.text(lon_s, lat_s + tick_h, "0", transform=ccrs.PlateCarree(), ha='center', va='bottom')
    ax.text(lon_e, lat_s + tick_h, f"{length_km} km", transform=ccrs.PlateCarree(), ha='center', va='bottom')

def add_north_arrow(ax, location=(0.90, 0.90), size=0.1):
    x0, x1, y0, y1 = ax.get_extent(ccrs.PlateCarree())
    lon, lat = x0 + (x1-x0)*location[0], y0 + (y1-y0)*location[1]
    h, w = (y1-y0)*size, (y1-y0)*size*0.25
    kw = dict(transform=ccrs.PlateCarree(), color='k', zorder=25)
    ax.add_patch(Polygon([(lon, lat), (lon-w, lat-h), (lon, lat-h*0.7)], fc='k', **kw))
    ax.add_patch(Polygon([(lon, lat), (lon, lat-h*0.7), (lon+w, lat-h)], fc='w', ec='k', lw=0.6, **kw))
    ax.text(lon, lat + (y1-y0)*0.01, 'N', ha='center', va='bottom', transform=ccrs.PlateCarree(), zorder=26)

CONFIG_PATH = Path("/content/drive/MyDrive/Southeast-Asia-XCO2-STK/src/config.yaml")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)


PROJECT_DIR = Path(config['project_root'])

PROC_DIR = PROJECT_DIR / config['paths']['output']['trend']
PROC_DIR.mkdir(parents=True, exist_ok=True)  # Ensure output directory exists

gis_dir = PROJECT_DIR / config['paths']['gis']

cams_nc = PROJECT_DIR / config['paths']['data']['cams_nc']
stk_nc = PROJECT_DIR / config['paths']['output']['stk_nc']

DOC_DIR = PROJECT_DIR / config['paths']['docs']
DOC_DIR.mkdir(parents=True, exist_ok=True)

REGIONS = {}
for name, attrs in config['regions'].items():
    shp_path = gis_dir / attrs['shp']
    gdf = gpd.read_file(shp_path) if shp_path.exists() else gpd.GeoDataFrame({'geometry': []}, crs="EPSG:4326")
    REGIONS[name] = (gdf, attrs['color'])


CITIES = config['cities']
STATIONS = config['stations']
LABELS = [(lbl['lon'], lbl['lat'], lbl['text'], lbl['color']) for lbl in config['labels']]
SITE_META = {station['site']: station['meta'] for station in STATIONS}

print(f"Configuration loaded successfully!")
print(f"Target output directory: {PROC_DIR}")
print(f"Loaded {len(CITIES)} cities and {len(STATIONS)} observation stations.")

# Geographic location of the study area

In [ ]:
import os
import re
import zipfile
import urllib.request
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import rasterio
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib.lines as mlines
import matplotlib.gridspec as gridspec
from rasterio.enums import Resampling
from matplotlib.patches import Polygon, Circle, Patch, ConnectionPatch
from shapely.geometry import box

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

warnings.filterwarnings("ignore")

COLORS = {
    "south": '#c96a6a',
    "indo":  '#6aae75',
    "malay": '#a8a8a8',
    "ocean": '#edf3f8',
    "land":  '#f7f7f2',
    "coast": '#546e7a',
    "border": '#b8c2c8'
}

EXTENT_REGIONAL = [92, 143, -12, 32]
lon_min, lon_max, lat_min, lat_max = EXTENT_REGIONAL

def get_online_ne_basemap():
    save_dir, tif_name = "./ne_basemap_cache", "HYP_50M_SR_W.tif"
    tif_path = os.path.join(save_dir, tif_name)
    url = "https://naturalearth.s3.amazonaws.com/50m_raster/HYP_50M_SR_W.zip"

    if not os.path.exists(tif_path):
        os.makedirs(save_dir, exist_ok=True)
        zip_path = tif_path.replace(".tif", ".zip")
        print(f"⏳ Downloading high-definition base map (approx. 20MB)...")
        urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(save_dir)
        os.remove(zip_path)
    return tif_path

def add_local_tif_basemap(ax, tif_path, zorder=0, alpha=1.0, downsample=1):
    if not tif_path or not os.path.exists(tif_path):
        ax.stock_img()
        return
    with rasterio.open(tif_path) as src:
        bounds = src.bounds
        out_shape = (src.count, int(src.height/downsample), int(src.width/downsample))
        img = src.read(out_shape=out_shape, resampling=Resampling.bilinear)
        rgb = np.transpose(img[:3], (1, 2, 0)).astype(np.float32) / 255.0
        ax.imshow(rgb, origin="upper", extent=[bounds.left, bounds.right, bounds.bottom, bounds.top],
                  transform=ccrs.PlateCarree(), zorder=zorder, alpha=alpha)


def add_dual_label(ax, lon, lat, zh, en, size_zh=10, size_en=9, color='black', dy=-1.0):
    effects = [pe.withStroke(linewidth=2.2, foreground="white")]
    kwargs = dict(transform=ccrs.PlateCarree(), color=color, ha='center', va='center', zorder=25, path_effects=effects)
    ax.text(lon, lat + dy, en, fontsize=size_en, **kwargs)

def add_city_markers(ax, cities, size=6):
    effects = [pe.withStroke(linewidth=1.5, foreground="white")]
    for city in cities:
        lon, lat, en = city["lon"], city["lat"], city["en"]
        ax.plot(lon, lat, 'o', color='red', mec='black', ms=2, lw=0.35, transform=ccrs.PlateCarree(), zorder=35)
        ax.text(lon-0.8, lat + 0.8, en, transform=ccrs.PlateCarree(), fontsize=size, ha='left', va='bottom',
                zorder=35, path_effects=effects)

def add_obs_stat(ax, stations, fontsize=8):
    effects = [pe.withStroke(linewidth=1.2, foreground="white")]
    for st in stations:
        lon, lat, name = st['lon'], st['lat'], st['site']
        st_type = st.get('type', 'wdcgg').lower()
        label = re.sub(r'\d+', '', str(name))

        marker, color, s, z = ('*', 'red', 110, 34) if st_type == 'tccon' else ('^', 'red', 45, 32)
        ax.scatter(lon, lat, marker=marker, color=color, edgecolors='black', s=s, transform=ccrs.PlateCarree(), zorder=z)

        Dxy = 1 if label in ["HAT", "HKG"] else 0
        ax.text(lon + 0.5 + Dxy, lat - 0.5 - Dxy, label, transform=ccrs.PlateCarree(),
                fontsize=fontsize, color='black', zorder=z+1, path_effects=effects)

    legend_elements = [
        mlines.Line2D([0], [0], color='w', marker='*', markerfacecolor='red', markeredgecolor='black', markersize=10, label='TCCON'),
        mlines.Line2D([0], [0], color='w', marker='^', markerfacecolor='red', markeredgecolor='black', markersize=7, label='WDCGG')
    ]
    leg = ax.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.11, 0.1), fontsize=fontsize-1,
                    frameon=True, facecolor='white', edgecolor='black', framealpha=1.0, columnspacing=0.8, handletextpad=0.2)
    leg.get_frame().set_linewidth(0.6)


fig = plt.figure(figsize=(10, 5), dpi=300)
gs = gridspec.GridSpec(1, 2, width_ratios=[0.5, 1], wspace=0.05)

ax1 = fig.add_subplot(gs[0], projection=ccrs.Orthographic(110, 15))
ax1.set_global()
ax1.add_patch(Circle((0.5, 0.5), 0.5, transform=ax1.transAxes, facecolor='#edf3f8', zorder=-1))
ax1.add_feature(cfeature.LAND, facecolor='#f7f7f2', edgecolor='none', zorder=0)
ax1.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, edgecolor='#546e7a', zorder=4)
ax1.add_feature(cfeature.COASTLINE, lw=0.5, ec='#546e7a', zorder=3)

region_patches = []
for label, (gdf, col) in REGIONS.items():
    if not gdf.empty:
        gdf.plot(ax=ax1, facecolor=col, edgecolor='#444444',
                 linewidth=0.3, transform=ccrs.PlateCarree(), zorder=2)
        region_patches.append(Patch(facecolor=col, edgecolor='#444444', label=label))

if region_patches:
    ax1.legend(handles=region_patches,
               loc='lower left',
               bbox_to_anchor=(0.05, -0.2), # Adjust position below the globe
               frameon=False,
               fontsize=8,
               ncol=1) # If too crowded, change to ncol=3 for horizontal arrangement

gdf_extent = gpd.GeoDataFrame({'geometry': [box(lon_min, lat_min, lon_max, lat_max)]}, crs="EPSG:4326")
gdf_extent.plot(ax=ax1, facecolor='red', alpha=0.1, edgecolor='red', linewidth=1.0, linestyle='--', transform=ccrs.PlateCarree(), zorder=5)
ax1.spines['geo'].set_visible(False)

ax2 = fig.add_subplot(gs[0, 1], projection=ccrs.PlateCarree())
ax2.set_extent(EXTENT_REGIONAL, crs=ccrs.PlateCarree())
add_local_tif_basemap(ax2, get_online_ne_basemap(), zorder=0, downsample=2)
ax2.add_feature(cfeature.COASTLINE.with_scale('50m'), lw=0.7, ec='black', zorder=10)
ax2.add_feature(cfeature.BORDERS.with_scale('50m'), lw=0.5, ec='0.3', zorder=10)

shp_file = shpreader.natural_earth(resolution='50m', category='cultural', name='admin_1_states_provinces')
reader = shpreader.Reader(shp_file)
provinces_cn = [rec.geometry for rec in reader.records() if rec.attributes['admin'] == 'China']
ax2.add_geometries(provinces_cn, crs=ccrs.PlateCarree(), edgecolor='gray', linewidth=0.3, facecolor='none', zorder=12)


add_city_markers(ax2, CITIES, size=6)
add_obs_stat(ax2, STATIONS, fontsize=8)
for lon, lat, _, col in LABELS:
    add_dual_label(ax2, lon, lat, "", "", size_zh=9, color=col)


ax2.set_xticks([95, 105, 115, 125, 135], crs=ccrs.PlateCarree())
ax2.set_yticks([-10, 0, 10, 20, 30], crs=ccrs.PlateCarree())
ax2.xaxis.set_major_formatter(LongitudeFormatter())
ax2.yaxis.set_major_formatter(LatitudeFormatter())
ax2.tick_params(labelsize=8)
ax2.gridlines(lw=0.3, color='gray', alpha=0.2, ls='--')

add_scalebar(ax2, 1000, (0.25, 0.02))
add_north_arrow(ax2, location=(0.9, 0.9))

trans1 = ccrs.PlateCarree()._as_mpl_transform(ax1)
trans2 = ccrs.PlateCarree()._as_mpl_transform(ax2)
kw_line = dict(axesA=ax1, axesB=ax2, color='#9c4c4c', lw=1.0, linestyle='--', zorder=100, clip_on=False)

fig.add_artist(ConnectionPatch(xyA=(lon_max, lat_max), xyB=(lon_min, lat_max), coordsA=trans1, coordsB=trans2, **kw_line))
fig.add_artist(ConnectionPatch(xyA=(lon_max, lat_min), xyB=(lon_min, lat_min), coordsA=trans1, coordsB=trans2, **kw_line))

for s in ax2.spines.values():
    s.set_linewidth(1.0)

png_fn = DOC_DIR / "location.png"
plt.savefig(png_fn, dpi=300, bbox_inches='tight')
plt.show()

#Pixel-wise calculation of the multi-year mean and interannual change rate.

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import linregress

def calc_trend(y, years):
    y = np.asarray(y, dtype=float)
    years = np.asarray(years, dtype=float)
    mask = np.isfinite(y) & np.isfinite(years)
    if mask.sum() < 5:
        return np.nan
    slope, _, _, _, _ = linregress(years[mask], y[mask])
    return slope


var_name = "xco2"
extent = [92, 143, -12, 32]
processed_results = {}

for nc_path in [cams_nc, stk_nc]:
    label = "CAMS" if "cams" in nc_path.name.lower() else "STK"
    print(f"Processing {label} ...")

    with xr.open_dataset(nc_path) as ds:
        ds = ds.copy()
        ds["time"] = pd.to_datetime(ds["time"].values)
        da = ds[var_name]

        da = da.sel(lon=slice(extent[0], extent[1]))

        if da.lat.values[0] < da.lat.values[-1]:
            lat_slice = slice(extent[2], extent[3])
        else:
            lat_slice = slice(extent[3], extent[2])
        da = da.sel(lat=lat_slice)

        da = da.where(np.isfinite(da), np.nan)
        da_year = da.groupby("time.year").mean("time", skipna=True)
        multi_year_mean = da_year.mean(dim="year", skipna=True)
        years_arr = da_year["year"].values.astype(float)

        trend_map = xr.apply_ufunc(
            calc_trend,
            da_year,
            years_arr,
            input_core_dims=[["year"], ["year"]],
            output_core_dims=[[]],
            vectorize=True,
            dask="parallelized",
            output_dtypes=[float]
        )

        ds_out = xr.Dataset(
            data_vars={
                "mean": multi_year_mean,
                "trend": trend_map
            },
            coords={
                "lat": da["lat"],
                "lon": da["lon"]
            }
        )

        out_nc = PROC_DIR / f"{label.lower()}_trend_year_2015_2024.nc"
        ds_out.to_netcdf(out_nc)
        processed_results[label] = ds_out
        print(f"Saved: {out_nc}")

#Calculate Regional Monthly Means


In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import regionmask

gdf_south = gpd.read_file(gis_dir / "South_China.shp") if (gis_dir / "South_China.shp").exists() else gpd.GeoDataFrame({'geometry': []}, crs="EPSG:4326")
gdf_indochina = gpd.read_file(gis_dir / "Indochina.shp") if (gis_dir / "Indochina.shp").exists() else gpd.GeoDataFrame({'geometry': []}, crs="EPSG:4326")
gdf_malay = gpd.read_file(gis_dir / "Malay_Archipelago.shp") if (gis_dir / "Malay_Archipelago.shp").exists() else gpd.GeoDataFrame({'geometry': []}, crs="EPSG:4326")
regions = {
    "Whole Area": None,
    "South China": gdf_south,
    "Indochina": gdf_indochina,
    "Malay Archipelago": gdf_malay
}

for nc_path in [cams_nc, stk_nc]:

    label = "CAMS" if "cams" in nc_path.name.lower() else "STK"
    print(f"\n--- Processing {label} (Monthly Only) ---")
    save_path = PROC_DIR / f"{label.lower()}_xco2_month_2015_2024.csv"

    ds = xr.open_dataset(nc_path)
    da = ds[var_name]

    da["time"] = pd.to_datetime(da["time"].values)
    da = da.sel(lon=slice(extent[0], extent[1]))
    lat_vals = da.lat.values
    lat_slice = slice(extent[2], extent[3]) if lat_vals[0] < lat_vals[-1] else slice(extent[3], extent[2])
    da = da.sel(lat=lat_slice)
    all_df = []
    for reg_name, gdf in regions.items():
        print(f"  Extracting: {reg_name}")

        if reg_name == "Whole Area":
            da_masked = da
        else:
            if gdf is None or gdf.empty:
                continue
            mask = regionmask.mask_3D_geopandas(gdf, da.lon, da.lat)
            da_masked = da.where(mask.isel(region=0))


        spatial_mean = da_masked.mean(dim=["lat", "lon"], skipna=True)

        monthly_series = (
            spatial_mean.resample(time="ME")
            .mean()
            .to_series()
        )

        df_region = monthly_series.to_frame(name=f"{reg_name}")
        all_df.append(df_region)

    if all_df:
      df_final = pd.concat(all_df, axis=1)


      df_final.to_csv(save_path)
      print(f"✅ File saved: {save_path}")
print("\nAll tasks completed!")

# Plot Map of Spatiotemporal_Patterns_of_XCO2 for CAMS and STK

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.dates as mdates
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pathlib import Path
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from scipy.stats import linregress
from matplotlib.gridspec import GridSpec
from mpl_toolkits.axes_grid1.inset_locator import inset_axes


def setup_geo_axis(ax, extent):
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor="#F7F7F7", zorder=0)
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), linewidth=0.5, zorder=2)
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=0.3, zorder=2)

    ax.set_xticks(np.arange(100, 141, 10), crs=ccrs.PlateCarree())
    ax.set_yticks(np.arange(-10, 31, 10), crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(labelsize=8)

    ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=False, linewidth=0.3,
                 color="gray", alpha=0.4, linestyle="--")

def add_panel_label(ax, label):
    ax.text(0.45, 1.05, label, transform=ax.transAxes,
            ha="left", va="bottom", fontsize=10)



fig = plt.figure(figsize=(8, 6.2), dpi=300)

# Master Grid: 1 row, 2 columns. Left for time series plots (38%), right for map group (62%)
# wspace=0.15 ensures distance between time series plots and mean maps
gs_master = GridSpec(nrows=1, ncols=2, figure=fig, width_ratios=[0.30, 0.7], wspace=0.1)

# Left Grid: 4 rows, 1 column (Time series plot 1, placeholder, Time series plot 2, placeholder)
gs_left = gs_master[0, 0].subgridspec(nrows=4, ncols=1, height_ratios=[1.0, 0.25, 1.0, 0.25], hspace=0.18)

# Right Grid: 4 rows, 2 columns (two maps side-by-side)
# wspace=0.02 ensures almost no gap between mean maps and trend maps
gs_right = gs_master[0, 1].subgridspec(nrows=4, ncols=2, height_ratios=[1.0, 0.25, 1.0, 0.25], hspace=0.18, wspace=0.05)

rows_data = ["cams", "stk"]
target_regions = ["Whole Area", "South China", "Indochina", "Malay Archipelago"]
region_colors = ["black", "#E41A1C", "#377EB8", "#4DAF4A"]
extent = [92, 143, -12, 32]

for r_idx, label in enumerate(rows_data):
    try:
        csv_path = PROC_DIR / f"{label}_xco2_month_2015_2024.csv"
        ts_df = pd.read_csv(csv_path, index_col=0, parse_dates=True)
        nc_path = PROC_DIR / f"{label.lower()}_trend_year_2015_2024.nc"
        ds_res = xr.open_dataset(nc_path)
    except Exception as e:
        print(f" Loading Error: {e}")
        continue

    row_main = r_idx * 2
    row_cbar = row_main + 1

    # ==========================
    # Column 1: Time Series (assigned to gs_left)
    # ==========================
    ax1 = fig.add_subplot(gs_left[row_main, 0])

    y_all = [ts_df[reg].dropna() for reg in target_regions if reg in ts_df.columns]
    if y_all:
        y_min = min([y.min() for y in y_all])
        y_max = max([y.max() for y in y_all])
        ax1.set_ylim(y_min - 1.5, y_max + 8)

    for reg, colr in zip(target_regions, region_colors):
        if reg in ts_df.columns:
            y_vals = ts_df[reg].values # Specify column for y_vals
            mask = np.isfinite(y_vals)
            # Ensure enough data points for linregress
            if mask.sum() > 1: # At least 2 points needed for linregress
                x_idx_masked = np.arange(len(y_vals))[mask]
                y_vals_masked = y_vals[mask]
                slope, _, _, _, _ = linregress(x_idx_masked, y_vals_masked)
                combined_label = f"{reg.replace('_', ' ')} ({slope*12:.2f} ppm yr$^{{-1}}$)"
                ax1.plot(ts_df.index, y_vals, color=colr, lw=1.0, label=combined_label)
            else:
                ax1.plot(ts_df.index, y_vals, color=colr, lw=1.0, label=f"{reg.replace('_', ' ')}") # Plot without trend if not enough data

    ax1.set_ylabel(f"{label.upper()} $XCO_2$ (ppm)", fontweight='bold', labelpad=4)
    ax1.xaxis.set_major_locator(mdates.YearLocator(2))
    ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax1.tick_params(axis='y', pad=2)
    ax1.legend(loc="upper left", frameon=False, fontsize=6.2, labelspacing=0.15, handletextpad=0.3, borderaxespad=0.3)
    add_panel_label(ax1, f"({chr(97 + r_idx * 3)})")

    # ==========================
    # Column 2: Mean Map (assigned to gs_right, column 0)
    # ==========================
    ax2 = fig.add_subplot(gs_right[row_main, 0], projection=ccrs.PlateCarree())
    setup_geo_axis(ax2, extent)
    vmin_m, vmax_m = np.nanpercentile(ds_res["mean"], [2, 98]) # Specified data variable
    im1 = ax2.pcolormesh(ds_res.lon, ds_res.lat, ds_res["mean"], cmap="Spectral_r", vmin=vmin_m, vmax=vmax_m, shading="auto") # Specified data variable

    add_panel_label(ax2, f"({chr(98 + r_idx * 3)})")
    add_north_arrow(ax2)
    add_scalebar(ax2)


    ax3 = fig.add_subplot(gs_right[row_main, 1], projection=ccrs.PlateCarree())
    setup_geo_axis(ax3, extent)


    vmin_t, vmax_t = np.nanpercentile(ds_res["trend"], [0.1, 99.9]) # Specified data variable
    im2 = ax3.pcolormesh(ds_res.lon, ds_res.lat, ds_res["trend"], cmap="RdBu_r", vmin=vmin_t, vmax=vmax_t, shading="auto") # Specified data variable

    add_panel_label(ax3, f"({chr(99 + r_idx * 3)})")
    add_north_arrow(ax3)
    add_scalebar(ax3)

    # ==========================
    # Colorbars (assigned to the colorbar placeholder row in gs_right)
    # ==========================
    # Mean Colorbar
    cax1_base = fig.add_subplot(gs_right[row_cbar, 0]) # Corrected indexing for gs_right
    cax1_base.axis("off")
    cax1 = inset_axes(cax1_base, width="60%", height="25%", loc="upper center", borderpad=0)
    cb1 = plt.colorbar(im1, cax=cax1, orientation="horizontal")
    cb1.set_label("Mean $XCO_2$ (ppm)", fontsize=7.5, labelpad=2)
    cb1.ax.tick_params(labelsize=7, pad=1, length=2)

    # Trend Colorbar
    cax2_base = fig.add_subplot(gs_right[row_cbar, 1]) # Corrected indexing for gs_right
    cax2_base.axis("off")
    cax2 = inset_axes(cax2_base, width="60%", height="25%", loc="upper center", borderpad=0)
    cb2 = plt.colorbar(im2, cax=cax2, orientation="horizontal")
    cb2.set_label("Trend (ppm yr$^{-1}$)", fontsize=7.5, labelpad=2)
    cb2.ax.tick_params(labelsize=7, pad=1, length=2)

    if 'ds_res' in locals():
        ds_res.close()

# =================================================================
# 5. Final rendering and output
# =================================================================
# Use subplots_adjust to fine-tune the outermost edges of the canvas
fig.subplots_adjust(left=0.01, right=0.98, top=0.92, bottom=0.06)

png_fn = DOC_DIR / "Spatiotemporal_Patterns_of_XCO2.png"
plt.savefig(png_fn, dpi=300, bbox_inches='tight')
print(png_fn)
plt.show()

#